# Costa del Sol Flood Risk - Notebook 3: Web Export (GeoJSON for MapLibre)

## 0. Setup

Reproyecta a EPSG:4326 y guarda un GeoJSON por capa en `data/web/`. No se recalcula ni se saca ningun indicador nuevo aqui - eso ya esta cerrado en `2_indicators.ipynb`, esto es puramente el paso de reproyectar y aligerar para la web.

In [1]:
import warnings
from pathlib import Path

import geopandas as gpd
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEB_DIR = PROJECT_ROOT / "data" / "web"
WEB_DIR.mkdir(parents=True, exist_ok=True)

WGS84 = "EPSG:4326"


def export_geojson(gdf, name, round_cols=None, decimals=2):
    """Reproyecta a WGS84 y guarda como data/web/<name>.geojson."""
    out = gdf.to_crs(WGS84).copy()
    if round_cols:
        for c in round_cols:
            if c in out.columns:
                out[c] = out[c].round(decimals)
    path = WEB_DIR / f"{name}.geojson"
    if path.exists():
        path.unlink()
    out.to_file(path, driver="GeoJSON")
    size_mb = path.stat().st_size / 1e6
    print(f"{name}.geojson: {len(out)} features, {size_mb:.2f} MB")
    return out


admin_municipios = gpd.read_file(PROCESSED_DIR / "admin_municipios_25830.gpkg")
print(f"{len(admin_municipios)} municipios cargados, CRS {admin_municipios.crs}")

62 municipios cargados, CRS EPSG:25830


## 1. Capa municipios (choropleth principal)

Un feature por municipio (62), con los dos carriles y los 4 periodos de retorno como propiedades separadas en vez de 4 capas SNCZI duplicadas - la idea ya apuntada en el plan (mismo patron que p1996/p1997 en el proyecto de demografia). Un selector de periodo en el cliente cambia que propiedad pinta el choropleth, el color siempre significa lo mismo.

Columnas por carril:
- Urbano: `eur_urbano_t10/t50/t100/t500`, `ead_eur_urbano`, mas detalle T100 (`n_edificios_inundados_t100`, `pct_area_construida_inundada_t100`, `valor_eur_m2`, `n_eventos` e `indemnizacion_total_eur` de CNIH).
- Agricola: `ha_agricola_t10/t50/t100/t500`, `ead_ha_agricola`, mas el desglose completo por tipo de cultivo en T100 (`ha_cultivo_<tipo>_t100`) para el popup, y el cultivo dominante como campo aparte.
- `is_grid` / `n_criterios`: que municipios tienen el grid de detalle (Fase 2, seccion 0).

In [2]:
PERIODOS = ["t10", "t50", "t100", "t500"]

urbano_multi = pd.read_csv(PROCESSED_DIR / "carril_urbano_ead_multiperiodo.csv")
urbano_t100 = pd.read_csv(PROCESSED_DIR / "carril_urbano_eur_en_riesgo_t100.csv")
agricola_multi = pd.read_csv(PROCESSED_DIR / "carril_agricola_ead_multiperiodo.csv")
agricola_t100 = pd.read_csv(PROCESSED_DIR / "carril_agricola_ha_en_riesgo_t100.csv")
grid_final = pd.read_csv(PROCESSED_DIR / "grid_municipios_final.csv")

# Mun_Code sale como int64 al leer CSV, pero en los gpkg es texto (object) - homogeneizar antes del merge
for _df in (urbano_multi, urbano_t100, agricola_multi, agricola_t100, grid_final):
    _df["Mun_Code"] = _df["Mun_Code"].astype(str)

# --- urbano: renombrar columnas por periodo, anadir el detalle T100 que no esta en el multiperiodo ---
urbano = urbano_multi[["Mun_Code"]].copy()
for p in PERIODOS:
    urbano[f"eur_urbano_{p}"] = urbano_multi[f"valor_en_riesgo_{p}_eur"]
urbano["ead_eur_urbano"] = urbano_multi["ead_valor_eur"]

urbano_detalle_t100 = urbano_t100[[
    "Mun_Code", "n_edificios_total", "n_edificios_inundados_t100",
    "pct_edificios_inundados_t100", "pct_area_construida_inundada_t100",
    "valor_eur_m2", "n_eventos", "indemnizacion_total_eur",
]]
urbano = urbano.merge(urbano_detalle_t100, on="Mun_Code", how="left")

# --- agricola: mismo patron, mas el desglose por cultivo en T100 ---
agricola = agricola_multi[["Mun_Code"]].copy()
for p in PERIODOS:
    agricola[f"ha_agricola_{p}"] = agricola_multi[f"ha_agricola_inundada_{p}"]
agricola["ead_ha_agricola"] = agricola_multi["ead_ha_agricola"]

crop_cols = [c for c in agricola_t100.columns if c not in ("Mun_Code", "Mun_Name", "ha_agricola_inundada_t100")]
crop_rename = {c: f"ha_cultivo_{c.replace(' ', '_')}_t100" for c in crop_cols}
agricola_crops = agricola_t100[["Mun_Code"] + crop_cols].rename(columns=crop_rename)
agricola_crops["cultivo_principal_t100"] = agricola_t100[crop_cols].idxmax(axis=1)
agricola_crops["ha_cultivo_principal_t100"] = agricola_t100[crop_cols].max(axis=1)
agricola = agricola.merge(agricola_crops, on="Mun_Code", how="left")

# --- grid de detalle ---
grid_flags = grid_final[["Mun_Code", "n_criterios"]].copy()
grid_flags["is_grid"] = True

# --- ensamblado final ---
municipios_web = admin_municipios.merge(urbano, on="Mun_Code", how="left")
municipios_web = municipios_web.merge(agricola, on="Mun_Code", how="left")
municipios_web = municipios_web.merge(grid_flags, on="Mun_Code", how="left")
municipios_web["is_grid"] = municipios_web["is_grid"].fillna(False)
municipios_web["n_criterios"] = municipios_web["n_criterios"].fillna(0).astype(int)

round_cols = [c for c in municipios_web.columns if c.startswith(("eur_urbano_", "ha_agricola_", "ha_cultivo_", "ead_"))]
municipios_web_out = export_geojson(municipios_web, "municipios", round_cols=round_cols, decimals=2)
municipios_web_out[["Mun_Code", "Mun_Name", "is_grid", "eur_urbano_t100", "ha_agricola_t100", "cultivo_principal_t100"]].head()

municipios.geojson: 62 features, 1.47 MB


,Mun_Code,Mun_Name,is_grid,eur_urbano_t100,ha_agricola_t100,cultivo_principal_t100
0,11004,Algeciras,False,71687756.0,1.51,Unclassified arable crop
1,11008,Los Barrios,False,NaN,79.99,Fruits
2,11013,Castellar de la Frontera,False,NaN,0.00,Barley
3,11021,Jimena de la Frontera,False,NaN,0.00,Barley
4,11022,La Línea de la Concepción,False,197551705.0,1.00,Unclassified arable crop


## 2. Grid de detalle - edificios (T100, 9 municipios)

Puntos de edificios inundados en T100 dentro de los 9 municipios del grid, con distancia al cauce y cota relativa (celda "## 3." de `2_indicators.ipynb`). Se activa en el mapa solo al hacer zoom sobre esos 9 municipios.

In [3]:
mdt02_edif = pd.read_csv(PROCESSED_DIR / "mdt02_distancia_cota_grid_t100.csv")
edif_pts = gpd.GeoDataFrame(
    mdt02_edif,
    geometry=gpd.points_from_xy(mdt02_edif["punto_x"], mdt02_edif["punto_y"]),
    crs="EPSG:25830",
).drop(columns=["punto_x", "punto_y"])

export_geojson(
    edif_pts, "edificios_grid_t100",
    round_cols=["dist_cauce_m", "cota_edificio_m", "cota_cauce_m", "cota_relativa_m"],
)

edificios_grid_t100.geojson: 31411 features, 9.18 MB


,Mun_Code,gml_id,dist_cauce_m,cota_edificio_m,cota_cauce_m,cota_relativa_m,geometry
0,11033,ES.SDGC.BU.1E11033M01GUAD,109.99,3.38,0.17,3.21,POINT (-5.41302 36.19487)
1,11033,ES.SDGC.BU.11033A00100019,456.62,16.75,11.92,4.83,POINT (-5.34845 36.32024)
2,11033,ES.SDGC.BU.11033A00200001,378.01,6.77,2.06,4.71,POINT (-5.31893 36.32473)
3,11033,ES.SDGC.BU.11033A00200002,395.89,8.68,5.16,3.52,POINT (-5.31896 36.32543)
4,11033,ES.SDGC.BU.11033A00200010,334.93,6.81,12.04,-5.22,POINT (-5.31612 36.32526)
...,...,...,...,...,...,...,...
31406,29901,ES.SDGC.BU.7562111UF6576S_PI.1379,153.14,3.75,3.11,0.64,POINT (-4.4829 36.64083)
31407,29901,ES.SDGC.BU.7562113UF6576S_PI.1380,114.54,3.67,3.10,0.57,POINT (-4.48325 36.64063)
31408,29901,ES.SDGC.BU.7562114UF6576S_PI.1381,109.39,4.68,3.05,1.63,POINT (-4.48328 36.64058)
31409,29901,ES.SDGC.BU.7663937UF6576S_PI.1431,77.47,3.50,3.02,0.48,POINT (-4.48391 36.64073)


## 3. Grid de detalle - agricola (T100, 9 municipios)

Centros de pixel de cultivo inundado en T100 (10m, celda "## 2.3"), mismas columnas que los edificios. Unos 220 mil puntos - no hace falta pre-agregar, tippecanoe controla la densidad por nivel de zoom.

In [4]:
mdt02_agri = pd.read_csv(PROCESSED_DIR / "mdt02_distancia_cota_agricola_grid_t100.csv")
agri_pts = gpd.GeoDataFrame(
    mdt02_agri,
    geometry=gpd.points_from_xy(mdt02_agri["x"], mdt02_agri["y"]),
    crs="EPSG:25830",
).drop(columns=["x", "y"])

export_geojson(
    agri_pts, "agricola_grid_t100",
    round_cols=["dist_cauce_m", "cota_suelo_m", "cota_cauce_m", "cota_relativa_m"],
)

agricola_grid_t100.geojson: 221428 features, 59.73 MB


,Mun_Code,cultivo,dist_cauce_m,cota_suelo_m,cota_cauce_m,cota_relativa_m,geometry
0,29054,Unclassified permanent crop,29.39,15.84,13.36,2.48,POINT (-4.61913 36.55988)
1,29054,Unclassified permanent crop,32.07,15.65,13.05,2.60,POINT (-4.61901 36.55979)
2,29054,Unclassified arable crop,110.09,2.89,5.70,-2.81,POINT (-4.63536 36.52974)
3,29054,Unclassified arable crop,115.47,2.76,5.64,-2.88,POINT (-4.63525 36.52974)
4,29054,Unclassified arable crop,96.45,3.78,5.78,-2.00,POINT (-4.63547 36.52965)
...,...,...,...,...,...,...,...
221423,11033,Fruits,44.35,1.14,0.27,0.87,POINT (-5.41612 36.1862)
221424,11033,Fruits,140.82,2.93,0.15,2.78,POINT (-5.41563 36.18477)
221425,11033,Fruits,148.07,2.51,0.15,2.36,POINT (-5.41563 36.18468)
221426,11033,Fruits,154.96,2.61,0.15,2.46,POINT (-5.41552 36.18468)


## 4. Zonas inundables SNCZI (4 periodos de retorno)

Los 4 gpkg se combinan en un solo GeoJSON con una columna `periodo` (t10/t50/t100/t500) - selector de periodo en el cliente en vez de 4 capas fijas, mismo patron que las columnas de municipios. Solo los campos utiles para el popup, se descartan los administrativos de SNCZI que no aportan al mapa.

In [5]:
snczi_parts = []
for p in PERIODOS:
    gdf = gpd.read_file(PROCESSED_DIR / f"snczi_{p}_25830.gpkg")
    gdf = gdf[["ID_ZONA", "ZONA", "RIO", "TIPO_ZONA", "geometry"]].copy()
    # simplificar en 25830 (metros) antes de reproyectar - sin esto salian polígonos
    # con hasta 160k vertices (resolucion del modelo hidraulico original), 450 MB para
    # 444 features, invisible a la escala de este mapa e imposible de subir al
    # contenedor cloud para tippecanoe (limite de 400 MB por fichero)
    gdf["geometry"] = gdf.geometry.simplify(10)
    gdf["periodo"] = p
    snczi_parts.append(gdf)

snczi_zonas = gpd.GeoDataFrame(pd.concat(snczi_parts, ignore_index=True), crs=snczi_parts[0].crs)
export_geojson(snczi_zonas, "snczi_zonas")

snczi_zonas.geojson: 444 features, 7.20 MB


,ID_ZONA,ZONA,RIO,TIPO_ZONA,geometry,periodo
0,ES060_ARPS_0059_T10_01,Arroyo Benagalbón,Arroyo Benagalbón,Q Periodo de retorno T10,"MULTIPOLYGON (((-4.24845 36.71095, -4.24861 36...",t10
1,ES060_ARPS_0230_T10_01,Arroyo Estanco,Arroyo Estanco,Q Periodo de retorno T10,"MULTIPOLYGON (((-4.28675 36.71491, -4.28688 36...",t10
2,ES060_ARPS_0228_T10_01,Arroyo Pollo Zamora,Arroyo Pollo Zamora,Q Periodo de retorno T10,"MULTIPOLYGON (((-4.30266 36.71445, -4.30269 36...",t10
3,ES060_ARPS_0060_T10_01,Arroyo Cañuelo,Arroyo Cañuelo,Q Periodo de retorno T10,"MULTIPOLYGON (((-4.23243 36.72815, -4.23221 36...",t10
4,ES060_ARPS_0227_T10_01,Arroyo Piletas,Arroyo Piletas,Q Periodo de retorno T10,"MULTIPOLYGON (((-4.30899 36.71389, -4.30906 36...",t10
...,...,...,...,...,...,...
439,ES063_ARPS_0024_T500_01,Arroyo Garganta de San Francisco,Arroyo Garganta de San Francisco,Q Periodo de retorno T500,"MULTIPOLYGON (((-5.68177 36.06932, -5.68206 36...",t500
440,ES063_ARPS_0023_T500_01,Río del Valle,Río del Valle,Q Periodo de retorno T500,"MULTIPOLYGON (((-5.69986 36.09723, -5.69995 36...",t500
441,ES063_ARPS_0022_T500_01,Arroyo Las Villas,Arroyo Las Villas,Q Periodo de retorno T500,"POLYGON ((-5.77768 36.09635, -5.77888 36.09535...",t500
442,ES063_ARPS_0021_T500_01,Arroyo Candalar,Arroyo Candalar,Q Periodo de retorno T500,"MULTIPOLYGON (((-5.81321 36.12931, -5.81324 36...",t500


## 5. Hidrografia de contexto

Superficie de tramos de curso (`hi_tramocurso_s`) y zonas humedas (`hi_zhumeda_s`) - capas de contexto para dibujar en el mapa, no entran en ningun calculo aqui (eso ya esta cerrado en `2_indicators.ipynb` con `hi_tramocurso_l`, la version en linea, excluyendo tramos `ficticio`).

In [6]:
cauces_s = gpd.read_file(PROCESSED_DIR / "hi_tramocurso_s_25830.gpkg")
cauces_s = cauces_s[["id_curso", "nombre", "tipo_curso", "persist", "canaliza", "geometry"]]
export_geojson(cauces_s, "hidrografia_cauces")

humedales = gpd.read_file(PROCESSED_DIR / "hi_zhumeda_s_25830.gpkg")
humedales = humedales[["id_zhum", "nombre", "tipo_zhum", "geometry"]]
export_geojson(humedales, "hidrografia_humedales")

hidrografia_cauces.geojson: 151 features, 4.27 MB
hidrografia_humedales.geojson: 2 features, 0.01 MB


,id_zhum,nombre,tipo_zhum,geometry
0,146,Caño Boca Ancha,20001,"POLYGON Z ((-5.89732 36.17806 2.5, -5.89712 36..."
1,1105,Laguna de Herrera,20002,"POLYGON Z ((-4.58926 37.09642 414.9, -4.58929 ..."


## 6. Limites administrativos y de cuenca

Solo el borde (`boundary`), no el relleno - para dibujar el contorno de comarca y de cuenca hidrologica sobre el choropleth de municipio, sin duplicar todo el poligono.

In [7]:
comarcas = gpd.read_file(PROCESSED_DIR / "admin_comarcas_25830.gpkg")
comarcas = comarcas[["Comarca_Code", "Comarca_Name", "geometry"]].copy()
comarcas["geometry"] = comarcas.geometry.boundary
export_geojson(comarcas, "limites_comarcas")

cuencas = gpd.read_file(PROCESSED_DIR / "hydro_cuencas_25830.gpkg")
cuencas = cuencas[["id_cuenca", "nombre", "geometry"]].copy()
cuencas["geometry"] = cuencas.geometry.boundary
export_geojson(cuencas, "limites_cuencas")

limites_comarcas.geojson: 3 features, 0.57 MB
limites_cuencas.geojson: 8 features, 0.47 MB


,id_cuenca,nombre,geometry
0,061098,None,"MULTILINESTRING ((-4.45518 36.66413, -4.45542 ..."
1,061104,None,"MULTILINESTRING ((-3.72529 36.87241, -3.7199 3..."
2,061097,RIO GUADALHORCE,"MULTILINESTRING ((-4.33723 37.12677, -4.33712 ..."
3,061096,None,"MULTILINESTRING ((-4.10563 36.7247, -4.10651 3..."
4,061099,RIO GUADIARO,"MULTILINESTRING ((-5.35281 36.74541, -5.35275 ..."
5,061100,None,"MULTILINESTRING ((-5.28072 36.27741, -5.28107 ..."
6,061095,RIO VELEZ,"MULTILINESTRING ((-4.35104 36.96862, -4.34651 ..."
7,062068,None,"MULTILINESTRING ((-5.90214 36.17749, -5.89389 ..."


## 7. Resumen

Eventos historicos CNIH: la fuente original (`.accdb`) no tiene coordenadas, solo el dato agregado por municipio, ya incluido como `n_eventos` / `indemnizacion_total_eur` dentro de `municipios.geojson` - no hay una capa de puntos propia para esto, no existe la geometria de origen.

Siguiente paso (fuera de este notebook): `convert.sh` con tippecanoe -> un unico `.pmtiles` (cada capa con su propio minzoom/maxzoom) y un `index.html` con MapLibre + PMTiles. No se ejecuta aqui todavia - hay que confirmar antes donde corre tippecanoe.

In [8]:
print("Ficheros en data/web/:")
for f in sorted(WEB_DIR.glob("*.geojson")):
    print(f" - {f.name}: {f.stat().st_size / 1e6:.2f} MB")

Ficheros en data/web/:
 - agricola_grid_t100.geojson: 59.73 MB
 - edificios_grid_t100.geojson: 9.18 MB
 - hidrografia_cauces.geojson: 4.27 MB
 - hidrografia_humedales.geojson: 0.01 MB
 - limites_comarcas.geojson: 0.57 MB
 - limites_cuencas.geojson: 0.47 MB
 - municipios.geojson: 1.47 MB
 - snczi_zonas.geojson: 7.20 MB
